# Fase 3 - Preparación de los datos
Deduplicación, unidades de partición, segmentación y split agrupado. Detalle en `docs/03_preparacion_datos.md`.

In [1]:
import os, sys, json
from pathlib import Path
while not Path('configs').exists(): os.chdir('..')
sys.path.insert(0, '.')
import pandas as pd
from IPython.display import SVG, Image, display
pd.set_option('display.width', 160)
T = Path('experiments/results/tables')

In [2]:
a = pd.read_csv('data/annotations/annotations.csv')
a.groupby('source').agg(imagenes=('image_id', 'size'), grupos=('dup_group', 'nunique'), unidades=('fruit_id', 'nunique'), conservadas=('keep', 'sum'))

,imagenes,grupos,unidades,conservadas
source,,,,
afruitdb,1584,1512,1512,1581
fruits360,2662,1363,74,2662
fruitvision,6978,6933,186,6895
kaggle,11325,4500,4500,6668
sultana,2400,2328,2328,2398
wgisd,369,367,40,369


In [3]:
s = pd.read_csv('data/processed/splits.csv')
pd.crosstab(s['source'], s['split'], margins=True)

split,external,test,train,val,All
source,,,,,
afruitdb,0,223,1119,224,1566
fruits360,0,120,2364,164,2648
fruitvision,0,1039,4826,1018,6883
kaggle,0,831,4163,817,5811
sultana,1495,102,488,94,2179
wgisd,0,48,274,46,368
All,1495,2363,13234,2363,19455


In [4]:
# verificación de fuga: ninguna unidad de partición en dos conjuntos
leak = s.groupby('fruit_id')['split'].nunique()
assert (leak == 1).all()
print('unidades:', len(leak), '| sin fuga entre conjuntos')

unidades: 7899 | sin fuga entre conjuntos


In [5]:
# las versiones anteriores conservan su partición
for v in ('v1', 'v2'):
    o = pd.read_csv(f'data/processed/splits_{v}.csv').merge(s, on='image_id', suffixes=('_o', ''))
    print(v, len(o), 'imágenes, mismo split:', bool((o['split_o'] == o['split']).all()))

v1 9556 imágenes, mismo split: True
v2 18669 imágenes, mismo split: True
